<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>3. Acciones sobre la estructura de un <i>DataFrame</i></font>  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>


---

In [ ]:
import pandas as pd
import numpy as np

pd.__version__

Para ilustrar esta libreta se utilizará principalmente el conjunto de datos sintético almacenado en `data/productos.csv`. Cada fila representa la compra y la venta de un producto por el dueño de la tienda. 

In [ ]:
types = {
    "Nombre": "str",
    "Producto": "str",
    "Apellido": "str",
    "Compra": "float64"
}
df_productos = pd.read_csv('data/productos.csv', dtype=types, index_col=0)

# Reordena las columnas
df_productos = df_productos.loc[:,['Nombre', 'Apellido','Producto','Compra','Venta']]
df_productos

---
## <font color="#5E5E5E"> 0. Cómo actúan las operaciones sobre los datos</font>


Antes de estudiar las operaciones que permiten modificar la estructura de un `DataFrame`, conviene conocer cómo gestiona *pandas* los cambios en los datos y qué significan dos parámetros que aparecen en muchos de sus métodos: `inplace` y `axis`.



### <font color="#5E5E5E"> Copy-on-Write: copia al escribir</font>

Desde la versión 3.0, *pandas* utiliza un mecanismo denominado **Copy-on-Write** (*copia al escribir*). Cuando una operación devuelve un nuevo `DataFrame` o una nueva `Series` a partir de otro objeto, el resultado **se comporta como una copia independiente**: modificar sus valores mediante operaciones de *pandas* no altera los del objeto original, ni viceversa. Sin embargo, esto no implica que todos los datos se copien inmediatamente en memoria. Ambos objetos pueden compartirlos internamente mientras no se modifiquen. Cuando una modificación requiere separar los datos, *pandas* copia la parte necesaria. Así se evitan copias innecesarias sin que los cambios en un objeto afecten al otro.



<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-triangle" aria-hidden="true"></i>
La asignación `df2 = df` **no crea otro DataFrame**: ambos nombres se refieren al mismo objeto. Por tanto, las modificaciones de ese objeto se observan a través de ambos nombres. En la siguiente sección se verá cómo crear una copia explícita.

</div>



### <font color="#5E5E5E"> El parámetro `inplace`</font>

Muchos de los métodos que modifican la estructura o los valores de un `DataFrame` o una `Series` aceptan un parámetro denominado `inplace`. Este determina **si se modifica el propio objeto o se devuelve el resultado en uno nuevo**:

* `inplace=False`: devuelve un nuevo objeto con el resultado de la operación y mantiene el original sin modificar. Es el comportamiento por defecto de los métodos que veremos.
* `inplace=True`: modifica el objeto sobre el que se invoca el método y devuelve `None`. 

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-triangle" aria-hidden="true"></i>
El parámetro `inplace` describe el efecto de la operación sobre el objeto, no cómo se gestiona la memoria. Utilizar `inplace=True` **no garantiza que se eviten copias internas** ni que la operación sea más rápida.

</div>




### <font color="#5E5E5E">El parámetro `axis`</font>


Un `DataFrame` tiene dos ejes: el índice de filas y el índice de columnas. Muchos métodos aceptan un parámetro denominado `axis` que permite indicar **sobre qué eje actúa la operación**:

* `axis=0` o `axis='index'`: el eje de las **filas**.
* `axis=1` o `axis='columns'`: el eje de las **columnas**.


<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Las formas numéricas y los nombres son equivalentes. Utilizar `'index'` y `'columns'` puede facilitar la lectura del código. El efecto concreto de `axis` depende de la operación, y se irá ilustrando en los ejemplos.

</div>

---

## <font color="#5E5E5E"> 1. Copia</font>
<br>

El método `DataFrame.copy()/Series.copy()` devuelve una copia del objeto. En *pandas* su uso es a veces clave para el control del estado del conjunto de datos (se verá más adelante). Este método admite un parámetro, `deep` que por defecto toma el valor `True` y provoca que se copien datos e índice. Si `deep=False` se crea una copia suprficial protegida por *copy-on-write* (solo se hace la copia física cuando es necesario, se verá después).

In [ ]:
df_productos_cp = df_productos.copy() 
df_productos_cp

---
## <font color="#5E5E5E"> 2. Reindexación</font>

Es posible establecer los valores del índice mediante el método `DataFrame.set_index()`.  Como se ha explicado anteriormente, el valor por defecto del parámetro `inplace` es `False`, por lo que genera un nuevo `DataFrame`. Si este método recibe el nombre de una columna, la utiliza como índice. El parámetro `drop` determina si se elimina la columna utilizada.

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: El valor por defecto del parámetro `drop` es `True`.  
</div>

In [ ]:
df_productos.set_index('Apellido', drop=True) # inplace=False, devuelve un nuevo DataFrame y df_productos permanece en su estado original
#df_productos

In [ ]:
df_productos

`DataFrame.set_index()` también acepta una colección de elementos **de longitud similar** al número de filas del `DataFrame`. Sin embargo, esta colección ha de ser un índice (`pd.Index`), una *Series*, un array *numpy*, etc. [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.set_index.html). 

In [ ]:
df_productos.set_index(pd.Index(['Tienda A', 'Tienda B', 'Tienda F', 'Tienda M', 'Tienda H'],  name='Tienda'), inplace=True)
#df_productos.set_index(np.array(['Tienda A', 'Tienda B', 'Tienda F', 'Tienda M', 'Tienda H']), inplace=False)

In [ ]:
df_productos

El método `DataFrame.reindex()` / `Series.reindex()` recibe una colección de etiquetas que se utiliza como **nuevo índice** y devuelve un nuevo objeto ajustado a ese índice. Las etiquetas determinan qué filas o elementos se conservan y en qué orden: los que no aparecen se descartan, y las etiquetas nuevas generan filas o elementos con valores perdidos, un valor arbitrario, o siguiendo alguna estrategia [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reindex.html) cuando el índice está ordenado monótonamente. Este método también permite reindexar las columnas con `axis=columns` (`1`).

In [ ]:
df_productos

In [ ]:
# Esto hace que las columnas Compar y Venta cambien de valor. 
#df_productos.reindex(['Tienda A',  'Tienda Z', 'Tienda B'], fill_value='----') 

# Alternativa se usa *Not a Number* pero se muestra '----')
df_productos.reindex(["Tienda A", "Tienda Z", "Tienda B"]).style.format(na_rep="----")
# df_productos.reindex(["Tienda A", "Tienda Z", "Tienda B"]).style.format(na_rep="----") # Equivalente

<div class="alert alert-block alert-danger">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i> Existen varias consideraciones en relación a esta función.

* El eje existente que se reindexa debe tener etiquetas únicas.
* `reindex()` no modifica el objeto original: devuelve un objeto nuevo con las etiquetas solicitadas.
* En pandas 3.0, su antiguo parámetro `copy` se ignora y está obsoleto.
* Las etiquetas objetivo pueden proporcionarse mediante una lista u otro objeto de tipo array o derivado.
</div>

El método `DataFrame.reindex_like()` toma como argumento un objeto y utiliza sus índices (filas y columnas) para reindexar. 

In [ ]:
df_otro = pd.DataFrame(
    {'Compra': [10, 20], 'Venta': [15, 30]},
    index=['Tienda A', 'Tienda B']
)
df_otro

In [ ]:
df_productos.reindex_like(df_otro)

Si solo  se quiere usar un índice, se puede hacer en dos pasos, accediendo al índice y utilizando `DataFrame.reindex()`. 

In [ ]:
df_productos.reindex(index=df_otro.index)


El método `DataFrame.reset_index()/Series.reset_index()` sustituye el índice actual por uno por defecto (de tipo `RangeIndex`). Conserva las etiquetas anteriores en una columna slavo que se indique, mediante `drop=True`, que se descarten.

In [ ]:
df_productos.reset_index(names="Tienda") # inplace=False. Como el índice no tenía nombre, se le asigna uno.

### <font color="#5E5E5E"> Alineación de índices </font>


El método `DataFrame.align()` permite ajustar dos `DataFrame` para que tengan **el mismo índice de filas, de columnas o ambos**. Devuelve **dos objetos**, uno por cada `DataFrame` original, con sus datos reorganizados según los índices resultantes.

El parámetro `join` determina qué etiquetas forman el nuevo índice:

* `'outer'`: todas las etiquetas presentes en cualquiera de los dos objetos. Es el valor por defecto.
* `'inner'`: solamente las etiquetas comunes a ambos.
* `'left'`: las etiquetas del objeto sobre el que se invoca el método.
* `'right'`: las etiquetas del objeto pasado como argumento.

El parámetro `axis` permite alinear solamente las filas (`axis=0`), solamente las columnas (`axis=1`) o ambos ejes (`axis=None`, por defecto). Las posiciones para las que un objeto no tiene datos se completan con valores perdidos.

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
`align()` no combina los datos en una sola tabla: devuelve **dos objetos alineados** y mantiene los originales sin modificar.

</div>

In [ ]:
df_productos

In [ ]:
df_ventas = df_productos.reindex(index=['Tienda B', 'Tienda F'], columns=['Producto', 'Venta'])
df_ventas

In [ ]:
df_productos_alineado, df_ventas_alineadas = df_productos.align(df_ventas, join='inner', axis=0)
df_productos_alineado

In [ ]:
df_ventas_alineadas

---
## <font color="#5E5E5E"> 3. Renombrado de índice y de columnas </font>

El índice y las columnas pueden renombrarse mediante el método `DataFrame.rename()`. Puede tomar como argumento un diccionario con las correspondencias, o una función de transformación [**( + inf)**](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html). Puede especificarse un diccionario para filas (`index`) y otro para columnas (`columns`).

In [ ]:
df_productos.rename(columns = {'Venta': 'PVP', 'Compra':'P. Compra'})  # inplace=False

En caso de tomar una función de transformación, es necesario especificar si ésta actúa sobre índices o sobre columnas mediante el parámetro `axis` (que por defecto toma el valor `index`). El siguiente ejemplo, establece el índice por defecto, renombra todos los valores del índice transformándolos en un `string` anteponiéndoles el carácter `'#'`. 

In [ ]:
df_productos.reset_index().rename(lambda x: '# '+str(x), axis='index') # inplace=False
#df_productos.reset_index().rename( index=lambda etiqueta: f"# {etiqueta}") # Equivalente

También es posible renombrar las columnas directamente asignando una colección de valores al campo `DataFrame.columns`. La colección de valores ha de contener un valor para cada columna. 

In [ ]:
df_productos.columns = ['NOMBRE', 'APELLIDO', 'PRODUCTO', 'COMPRA','VENTA']
df_productos

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Esta forma de renombrar *no* es un método, sino una asignación a un atributo del *DataFrame*. Por tanto, **el cambio persiste** en el propio `DataFrame`. 
</div>

Por otra parte, `DataFrame.rename_axis()` asigna o modifica el nombre del índice de filas o del índice de columnas.

In [ ]:
df_productos.rename_axis('Tienda', axis=0, inplace=False)

El método `DataFrame.set_axis()` permite asignar nuevas etiquetas al índice (o columnas).

In [ ]:
df_productos.set_axis(['Nombre', 'Apellido', 'Producto', 'P.Compra', 'PVP'], axis=1)

---
## <font color="5E5E5E"> 4. Reordenación de columnas y ordenación de filas </font>

Al acceder a una colección de columnas se genera un nuevo `DataFrame` en el que las columnas se colocan en el orden de la colección. 

In [ ]:
df_productos = df_productos[['PRODUCTO', 'NOMBRE',  'APELLIDO', 'COMPRA', 'VENTA']]
df_productos

El método `Series.sort_index()`/`DataFrame.sort_index()` ordena el eje (filas/columnas) según los valores de su índice. El parámetro `ascending` determina la dirección de la ordenación.

In [ ]:
df_productos.sort_index(axis='index', ascending=False, inplace=True)
df_productos

El método `Series.sort_values()` ordena la serie según sus valores. También acepta los parámetros `ascending` e `inplace`.

In [ ]:
df_productos['PRODUCTO'].sort_values(ascending=False, inplace=False) # No cambia el DataFrame porque se hace una copia.

El método `DataFrame.sort_values()` ordena el `DataFrame` secuencialmente en función de una o varias columnas. En este último caso,  se ordena progresivamente por cada una de ellas. Mediante el parámetro `ascending` se puede especificar el sentido de la ordenación para cada columna.  Por ejemplo, el siguiente código ordena ascendentemente el `DataFrame` `df_fifa` en función del campo _Age_ y, en segunda instancia para deshacer empates, descendentemente en función del campo _Overall_ (en realidad se hace al revés).

In [ ]:
df_fifa = pd.read_csv('./data/fifa19.csv', index_col=0).set_index('ID')
df_fifa

In [ ]:
df_fifa.sort_values(['Age', 'Overall'], ascending=[True, False]).head(10)

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
 El parámetro `kind` permite determinar el algoritmo de ordenación cuando se ordena una sola columna. En este caso, es interesante saber que `mergesort`  es un algoritmo de ordenación estable. Es decir, cuando ordena por un campo, para los elementos con un mismo valor de ese campo, preserva el orden relativo de los elementos anterior a la ordenación. `quicksort` por el contrario, no lo es.
</div>

---
## <font color="5E5E5E"> 5. Eliminación de filas y columnas </font>

El método `DataFrame.drop()` permite eliminar filas y columnas. `Series.drop()` elimina elementos de una serie según su índice. Ambos aceptan el parámetro `inplace`.

In [ ]:
df_productos.drop('Tienda B') # axis=0, inplace = False

In [ ]:
df_productos.drop(['COMPRA','VENTA'], axis=1) # inplace=False

Se pueden borrar __columnas__ también mediante la sentencia `del` de *python*. En este caso, la operación se hace *siempre* sobre el propio `DataFrame`.

In [ ]:
del df_productos['PRODUCTO']
df_productos

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: Esta función solamente permite borrar __columnas__, no filas. 
</div>

Otra función,  `pop` **devuelve y elimina** columnas en una misma operación.  En este caso, también, la operación se hace *siempre* sobre el propio `DataFrame`.

In [ ]:
df_productos.pop('APELLIDO')

In [ ]:
df_productos

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>